In [42]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import scipy
from itertools import combinations

In [43]:
sys.path.insert(0, "src")

from grn import grn

np.random.seed(5)


In [44]:
#manually adjust GRN hyperparameters in an attempt to build GRN with more non-additive effects
G = grn().add_structure(
    n=256,
    k=1,
    alpha=1e-99,
    beta=0.99,       # 1 - 1/r
    gamma=0.01,      # 1/r
    kappa=10,        # --w
    delta_in=100,
    delta_out=100,
)

In [45]:
G.simulate_rna(
    s=1e-4,
    dt=1e-2,
    tmax=20000,
    burnin=5000,
    tol=1e-3,
    save=True,
)
x_wt = np.zeros(256)
y_wt = G.rna

In [49]:
single_ko_expression = np.zeros((G.n, G.n))
single_ko_logfc = np.zeros((G.n, G.n))
single_ko_convergence = {}

for gene in tqdm(range(G.n), desc="Simulating single-gene KOs"):
    
    result = G.ko_nodes(
        genes=[gene],
        stats=("new_rna", 'logfc', 'convergence'),
        s=1e-4,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
    )
    
    single_ko_expression[gene, :] = result["new_rna"].flatten()
    single_ko_logfc[gene, :] = result["logfc"].flatten()
    single_ko_convergence[gene] = result['convergence']
x_single_ko = np.eye(G.n)
y_single_ko = single_ko_expression

Simulating single-gene KOs:   0%|          | 0/256 [00:00<?, ?it/s]

In [65]:
pairs = list(combinations(range(G.n), 2))

n_jobs = 64

def run_double_ko(pair):
    a, b = pair

    result = G.ko_nodes(
        genes=[a, b],
        stats=("new_rna", "logfc", 'convergence'),
        s=0,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
    )

    return (
        np.asarray(result["new_rna"]).flatten(),
        np.asarray(result["logfc"]).flatten(),
        np.asarray(result['convergence'])
    )


results = Parallel(
    n_jobs=64,
    pre_dispatch="n_jobs",
    prefer="processes",
    verbose=10,
)(
    delayed(run_double_ko)(pair)
    for pair in pairs
)

double_expression = np.asarray(
    [r[0] for r in results],
    dtype=np.float32
)

double_logfc = np.asarray(
    [r[1] for r in results],
    dtype=np.float32
)
double_convergence = np.asarray(
    [r[2] for r in results],
    dtype=np.float32
)



#now prepare as training data

pairs = np.array(
    pairs,
    dtype=np.int32
)

x_double_ko = np.zeros((pairs.shape[0], G.n))

#now fill x for doubleKO with 1s

rows = np.arange(pairs.shape[0])
x_double_ko[rows, pairs[:, 0]] = 1.0
x_double_ko[rows, pairs[:, 1]] = 1.0

y_double_ko = double_expression

[Parallel(n_jobs=64)]: Using backend LokyBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done   8 tasks      | elapsed:  1.2min
[Parallel(n_jobs=64)]: Done  21 tasks      | elapsed:  1.2min
[Parallel(n_jobs=64)]: Done  34 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done  49 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done  64 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done  81 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done  98 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 117 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 136 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 157 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 178 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 201 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 224 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 249 tasks      | elapsed:  1.3min
[Parallel(n_jobs=64)]: Done 274 tasks      | elapsed:  

# Lets train MLP on toy 256 gene GRN

In [77]:
seed = 0

import torch
from torch import nn
torch.manual_seed(seed)
device = torch.device('cuda')
#for more precision
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

In [29]:
#model implementation without dropout
model = nn.Sequential(nn.Linear(256, 512), nn.GELU(), nn.Linear(512, 512), nn.GELU(), nn.Linear(512, 256)).to(device)
linear = nn.Linear(256, 256).to(device)

#implement AdamW
import torch.optim as optim
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

In [78]:
#process data, standardise, put them on GPU

rng = np.random.default_rng(42)

n_pairs = pairs.shape[0]
nt, nv = int(0.5 * n_pairs), int(0.1 * n_pairs)

indices = rng.permutation(n_pairs)
test_idx = indices[:nt]
val_idx = indices[nt:nt + nv]
train_idx = indices[nt + nv:]

xtr = np.concatenate([x_wt[None, :], x_single_ko, x_double_ko[train_idx]], axis=0)
ytr = np.concatenate([y_wt[None, :], y_single_ko, y_double_ko[train_idx]], axis=0)

xv = x_double_ko[val_idx]
yv = y_double_ko[val_idx]

x_test = x_double_ko[test_idx]
y_test = y_double_ko[test_idx]

mean, scale = np.mean(ytr, axis=0), np.maximum(ytr.std(axis=0), 1e-6)

s_ytr = (ytr - mean) / scale

xtr_t   = torch.as_tensor(xtr, dtype=torch.float32, device=device)
s_ytr_t = torch.as_tensor(s_ytr, dtype=torch.float32, device=device)

xv_t    = torch.as_tensor(xv, dtype=torch.float32, device=device)
yv_t    = torch.as_tensor(yv, dtype=torch.float32, device=device)

x_test_t = torch.as_tensor(x_test, dtype=torch.float32, device=device)
y_test_t = torch.as_tensor(y_test, dtype=torch.float32, device=device)

mean_t  = torch.as_tensor(mean, dtype=torch.float32, device=device)
scale_t = torch.as_tensor(scale, dtype=torch.float32, device=device)

